# InceptionV3 — Check predictions after image changes

Use case: check whether a photo-tagging model behaves consistently after brightness changes, flipping or cropping. InceptionV3 combines feature processing at different scales.

Run the cells from top to bottom. Each notebook is independent.


## Step 1 — Install packages
Use Python 3.11 or 3.12 as a straightforward training environment, or Google Colab. Install into the active notebook kernel. Restart the kernel after the first installation if required. Internet is needed initially for packages, sample images/data and ImageNet weights. A GPU is optional; CPU runs can be slower. Model downloads can be hundreds of MB.


In [ ]:
%pip install tensorflow numpy matplotlib scikit-learn pillow


## Step 2 — Import libraries
TensorFlow runs the neural network. Keras provides the model API. NumPy handles arrays and Matplotlib displays pictures.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)


## Step 3 — Load a picture
We start with the same sample used in the classification demo.


In [ ]:
image_path = keras.utils.get_file(
    "sunflower.jpg",
    "https://storage.googleapis.com/download.tensorflow.org/example_images/592px-Red_sunflower.jpg"
)
# To use your own picture, replace image_path with a local filename.
original = keras.utils.load_img(image_path)
plt.figure(figsize=(6, 6))
plt.imshow(original)
plt.axis("off")
plt.show()


## Step 4 — Create realistic image variations
These changes mimic different lighting or framing. A crop may remove meaningful content, so identical predictions are not always expected.


In [ ]:
from PIL import ImageEnhance, ImageOps
width, height = original.size
variations = [original.convert("RGB"),
              ImageEnhance.Brightness(original.convert("RGB")).enhance(0.6),
              ImageOps.mirror(original.convert("RGB")),
              original.convert("RGB").crop((width//5, height//5, 4*width//5, 4*height//5))]
variation_names = ["Original", "Darker", "Flipped", "Center crop"]


## Step 5 — Predict each version
InceptionV3 expects 299 × 299 images and its preprocessing scales pixel values to [-1, 1].


In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input, decode_predictions
model = InceptionV3(weights="imagenet")
arrays = []
for picture in variations:
    arrays.append(keras.utils.img_to_array(picture.resize((299, 299))))
batch = preprocess_input(np.stack(arrays))
predictions = model.predict(batch, verbose=0)
decoded = decode_predictions(predictions, top=1)
plt.figure(figsize=(12, 4))
for i in range(len(variations)):
    label, score = decoded[i][0][1:]
    print(variation_names[i], label, float(score))
    plt.subplot(1, 4, i + 1)
    plt.imshow(variations[i])
    plt.title(f"{variation_names[i]}\n{label}: {score:.1%}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## Step 6 — Measure score changes
Track the original predicted class score across all variations. This is a sensitivity check on one image, not an overall robustness metric.


In [ ]:
original_class = int(np.argmax(predictions[0]))
original_label = decoded[0][0][1]
plt.bar(variation_names, predictions[:, original_class])
plt.ylabel("Score for the original predicted class")
plt.title(original_label)
plt.ylim(0, 1)
plt.show()


## Understand the result
A stable prediction on one image does not establish reliability. Repeat across a labeled test set and report accuracy per transformation. Scores are not calibrated probabilities of correctness.


## Practice
1. Change the input image or selected image index.
2. Explain the output in your own words.
3. Compare the result with the original image and record one limitation.


## References
- [Keras Applications](https://keras.io/api/applications/)
- [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html)
- [Keras Grad-CAM example](https://keras.io/examples/vision/grad_cam/)
